In [ ]:
from pathlib import Path
import json
import pandas as pd
import yaml
from tqdm.auto import tqdm
import os

import torch
import torch.nn.functional as F

import chg
from chg.misc.prompt_tokenizer import PromptTokenizer
from chg.misc import format_data

In [ ]:
project_root = Path(chg.__file__).parent.parent.parent
with open(project_root / "config.yaml") as f:
    config = yaml.safe_load(f)
    
directories = {k: Path(v) for k, v in config['directories'].items()}
os.environ['HF_HOME'] = str(directories['huggingface'])

from transformers import AutoTokenizer
from datasets import load_dataset

device = 0

In [ ]:
def create_dataset(tokenizer, task_name, questions, targets, instructions=None, example_set_size=10, num_examples=1,
                   question_prefix='Question: ', answer_prefix='Answer: ', sep='\n', seed=None, save_dir=None):
    df_prompts, text_tokens, loss_masks = format_data.create_dataset(
        tokenizer=tokenizer, 
        instructions=instructions,
        questions=questions,
        targets=targets,
        example_set_size=example_set_size,
        num_examples=num_examples,
        question_prefix=question_prefix,
        answer_prefix=answer_prefix,
        seed=seed,
        sep=sep)
    
    retval = [df_prompts]
    if save_dir is not None:
        df_prompts.to_parquet(save_dir / f'{task_name}.parquet')
    for split in ['train', 'validation']:
        split_mask = df_prompts.split == split
        dataset = {
            'text_tokens': text_tokens[split_mask],
            'loss_masks': loss_masks[split_mask],
        }
        retval.append(dataset)
        if save_dir is not None:
            save_path = save_dir / f'{task_name}_{split}.pt'
            torch.save(dataset, save_path)
    return tuple(retval)

In [ ]:
instructions = {
    'antonym': 'Given an input word, generate the word with opposite meaning.',
    'capitalize': 'Given an input word, generate the same word with a capital first letter.',
    'country-capital': 'Given a country name, generate the capital city.',
    'english-french': 'Given an English word, generate the French translation of the word.',
    'present-past': "Given a verb in the present tense, generate the verb's simple past inflection.",
    'singular-plural': 'Given a singular noun, generate its plural inflection.'
}

In [ ]:
seed = 0

model_names = [
    'meta-llama/Llama-3.2-3B-Instruct',
]

for model_name in tqdm(model_names):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    
    for task_name, instruction in instructions.items():
        save_dir = directories['save'] / f'datasets/function_vectors/{model_name}'
        save_dir.mkdir(parents=True, exist_ok=True)
        parquet_name = f'prompts.parquet'
        if os.path.exists(save_dir / parquet_name):
            print(f'Skipping {model_name} {task_name}, already exists.')
            continue

        print(f'Processing {model_name} {task_name}...')
        path = directories['repo'] / f'data/function_vectors/{task_name}.json'
        with open(path) as f:
            data = json.load(f)
        df = pd.DataFrame(data)
        
        df_kshot, trainset_kshot, valset_kshot = create_dataset(tokenizer, f'{task_name}_10shot', df['input'], df['output'], example_set_size=10, num_examples=10, 
                    question_prefix='Q: ', answer_prefix='A: ', sep='\n\n', seed=seed, save_dir=save_dir)
        df_inst, trainset_inst, valset_inst = create_dataset(tokenizer, f'{task_name}_inst', df['input'], df['output'], instructions=instruction, example_set_size=10, num_examples=1, 
                    question_prefix='Q: ', answer_prefix='A: ', sep='\n\n', seed=seed, save_dir=save_dir)
        print()

# Create LOO Datasets

Only create LOO For Llama 3.2-3B-Instruct

In [ ]:
fv_dirname = directories['save'] / f'datasets/function_vectors/meta-llama/Llama-3.2-3B-Instruct'
loo_dirname = directories['save'] / f'datasets/function_vectors_loo/meta-llama/Llama-3.2-3B-Instruct'
loo_dirname.mkdir(exist_ok=True, parents=True)

# Load datasets
kshot_paths = sorted(fv_dirname.glob('*_10shot_train.pt'))
inst_paths = sorted(fv_dirname.glob('*_inst_train.pt'))
task_names = [p.name.split('_')[0] for p in kshot_paths]

kshot_datasets = [torch.load(p) for p in kshot_paths]
inst_datasets = [torch.load(p) for p in inst_paths]

# Helper to pad list of tensors
def pad_cat(tensors, pad_val):
    max_len = max(t.size(1) for t in tensors)
    padded = [
        F.pad(t, (0, max_len - t.size(1)), value=pad_val)
        for t in tensors
    ]
    return torch.cat(padded, dim=0)

for leave_idx in range(len(task_names)):
    leave_out_task = task_names[leave_idx]
    kshot_train = [d for i, d in enumerate(kshot_datasets) if i != leave_idx]
    inst_train = [d for i, d in enumerate(inst_datasets) if i != leave_idx]

    for setup, pos_source, neg_source in [('10shot', kshot_train, inst_train), ('inst', inst_train, kshot_train)]:
        pos_text_tokens = pad_cat([d['text_tokens'] for d in pos_source], tokenizer.eos_token_id)
        pos_loss_masks = pad_cat([d['loss_masks'] for d in pos_source], 0)
        neg_text_tokens = pad_cat([d['text_tokens'] for d in neg_source], tokenizer.eos_token_id)
        neg_loss_masks = pad_cat([d['loss_masks'] for d in neg_source], 0)

        output = {
            'positive_text_tokens': pos_text_tokens,
            'positive_loss_masks': pos_loss_masks,
            'negative_text_tokens': neg_text_tokens,
            'negative_loss_masks': neg_loss_masks,
        }
            
        save_path = loo_dirname / f'{leave_out_task}_{setup}.pt'
        print(f'Saving {save_path}')
        torch.save(output, save_path)